In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [2]:

i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.
dir = r"E:\Downloads\Telegram Desktop\plant_disease\all\Crop Diseases\wheat"
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers
num = len(class_names)

base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm

x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num, activation = "softmax")(x)
model = models.Model(inputs, outputs)
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)


Found 1826 files belonging to 2 classes.
Using 1461 files for training.
Found 1826 files belonging to 2 classes.
Using 365 files for validation.
Classes: ['Wheat___Brown_Rust', 'Wheat___Yellow_Rust']
Train batches: 46
Val batches: 6
Test batches: 6
Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7940 - loss: 0.4813

46/46 ━━━━━━━━━━━━━━━━━━━━ 80s 1s/step - accuracy: 0.8700 - loss: 0.3132 - val_accuracy: 0.9827 - val_loss: 0.1312 - learning_rate: 0.0010
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9517 - loss: 0.1316

46/46 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - accuracy: 0.9493 - loss: 0.1275 - val_accuracy: 0.9884 - val_loss: 0.0830 - learning_rate: 0.0010
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 959ms/step - accuracy: 0.9686 - loss: 0.0824

46/46 ━━━━━━━━━━━━━━━━━━━━ 60s 1s/step - accuracy: 0.9699 - loss: 0.0784 - val_accuracy: 0.9884 - val_loss: 0.0590 - learning_rate: 0.0010
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 983ms/step - accuracy: 0.9844 - loss: 0.0516

46/46 ━━━━━━━━━━━━━━━━━━━━ 52s 1s/step - accuracy: 0.9836 - loss: 0.0477 - val_accuracy: 0.9942 - val_loss: 0.0362 - learning_rate: 0.0010
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.9808 - loss: 0.0512 - val_accuracy: 0.9769 - val_loss: 0.0474 - learning_rate: 0.0010
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9802 - loss: 0.0486 - val_accuracy: 0.9827 - val_loss: 0.0401 - learning_rate: 0.0010
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.9870 - loss: 0.0459 - val_accuracy: 0.9827 - val_loss: 0.0366 - learning_rate: 3.0000e-04


In [3]:
test_loss, test_accuracy = model.evaluate(test)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.9792 - loss: 0.0709
Test Loss: 0.0709
Test Accuracy: 97.92%


In [4]:
model.save("wheat.keras")